In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('D:\COEP\FintelliQ\data\processed\churn_cleaned.csv')
print(df.shape)
print(df.head())


(10000, 15)
   CreditScore Geography  Gender  Age  Tenure    Balance  NumOfProducts  \
0          619    France       0   42       2       0.00              1   
1          608     Spain       0   41       1   83807.86              1   
2          502    France       0   42       8  159660.80              3   
3          699    France       0   39       1       0.00              2   
4          850     Spain       0   43       2  125510.82              1   

   HasCrCard  IsActiveMember  EstimatedSalary  Exited  Complain  \
0          1               1        101348.88       1         1   
1          0               1        112542.58       0         1   
2          1               0        113931.57       1         1   
3          0               0         93826.63       0         0   
4          1               1         79084.10       0         0   

   Satisfaction Score Card Type  Point Earned  
0                   2   DIAMOND           464  
1                   3   DIAMOND       

In [3]:
# Complain ko alag save karo — baad mein business insight ke liye
complain_df = df[['Complain', 'Exited']].copy()

# Ab main dataframe se drop karo
df.drop(columns=['Complain', 'Satisfaction Score', 
                 'Point Earned', 'HasCrCard', 
                 'Card Type', 'EstimatedSalary'], inplace=True)

print("Columns after drop:")
print(df.columns.tolist())

Columns after drop:
['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'IsActiveMember', 'Exited']


In [ ]:
# 1. IsZeroBalance
# EDA mein dekha — zero balance alag group hai
df['IsZeroBalance'] = (df['Balance'] == 0).astype(int)

# 2. AgeGroup
# EDA mein dekha — 40-60 high risk hai
df['AgeGroup'] = pd.cut(df['Age'],
                         bins=[0, 30, 40, 50, 60, 100],
                         labels=['Young', 'Mid', 'Senior', 'Elder', 'Old'])

# 3. IsHighRiskProduct
# EDA mein dekha — 3-4 products = 82-100% churn
df['IsHighRiskProduct'] = (df['NumOfProducts'] >= 3).astype(int)

# 4. IsGermany
# EDA mein dekha — Germany 2x churn
df['IsGermany'] = (df['Geography'] == 'Germany').astype(int)

# 5. BalanceToSalaryRatio — drop kiya EstimatedSalary but ratio useful hai
# Reload salary just for this feature
df_original = pd.read_csv('D:\COEP\FintelliQ\data\processed\churn_cleaned.csv')
df['BalanceToSalaryRatio'] = df['Balance'] / (df_original['EstimatedSalary'] + 1)

# 6. IsInactiveSenior — interaction term
# EDA mein dekha — inactive + age 50+ = 81-86% churn
df['IsInactiveSenior'] = ((df['IsActiveMember'] == 0) & 
                           (df['Age'] >= 50)).astype(int)

# 7. IsFemaleGermany — interaction term
# EDA mein dekha — female + Germany = 37.6% churn
df['IsFemaleGermany'] = ((df['Gender'] == 0) & 
                          (df['Geography'] == 'Germany')).astype(int)



print("New features added:")
print(df.columns.tolist())
print(df.shape)

New features added:
['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'IsActiveMember', 'Exited', 'IsZeroBalance', 'AgeGroup', 'IsHighRiskProduct', 'IsGermany', 'BalanceToSalaryRatio', 'IsInactiveSenior', 'IsFemaleGermany']
(10000, 16)


In [6]:
# Naye features ka quick check
print("IsZeroBalance distribution:")
print(df.groupby('IsZeroBalance')['Exited'].mean().round(3) * 100)

print("\nAgeGroup churn rate:")
print(df.groupby('AgeGroup', observed=True)['Exited'].mean().round(3) * 100)

print("\nIsHighRiskProduct churn rate:")
print(df.groupby('IsHighRiskProduct')['Exited'].mean().round(3) * 100)

print("\nIsInactiveSenior churn rate:")
print(df.groupby('IsInactiveSenior')['Exited'].mean().round(3) * 100)

print("\nIsFemaleGermany churn rate:")
print(df.groupby('IsFemaleGermany')['Exited'].mean().round(3) * 100)

IsZeroBalance distribution:
IsZeroBalance
0    24.1
1    13.8
Name: Exited, dtype: float64

AgeGroup churn rate:
AgeGroup
Young      7.5
Mid       12.1
Senior    34.0
Elder     56.2
Old       24.8
Name: Exited, dtype: float64

IsHighRiskProduct churn rate:
IsHighRiskProduct
0    18.2
1    85.9
Name: Exited, dtype: float64

IsInactiveSenior churn rate:
IsInactiveSenior
0    17.2
1    82.2
Name: Exited, dtype: float64

IsFemaleGermany churn rate:
IsFemaleGermany
0    18.1
1    37.6
Name: Exited, dtype: float64


In [8]:
df.to_csv('../data/churn_features.csv', index=False)
print(f"Saved: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nFinal columns:")
print(df.columns.tolist())

Saved: 10000 rows x 16 columns

Final columns:
['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'IsActiveMember', 'Exited', 'IsZeroBalance', 'AgeGroup', 'IsHighRiskProduct', 'IsGermany', 'BalanceToSalaryRatio', 'IsInactiveSenior', 'IsFemaleGermany']


In [9]:
print("IsZeroBalance distribution:")
print(df.groupby('IsZeroBalance')['Exited'].mean().round(3) * 100)

print("\nAgeGroup churn rate:")
print(df.groupby('AgeGroup', observed=True)['Exited'].mean().round(3) * 100)

print("\nIsHighRiskProduct churn rate:")
print(df.groupby('IsHighRiskProduct')['Exited'].mean().round(3) * 100)

print("\nIsInactiveSenior churn rate:")
print(df.groupby('IsInactiveSenior')['Exited'].mean().round(3) * 100)

print("\nIsFemaleGermany churn rate:")
print(df.groupby('IsFemaleGermany')['Exited'].mean().round(3) * 100)

IsZeroBalance distribution:
IsZeroBalance
0    24.1
1    13.8
Name: Exited, dtype: float64

AgeGroup churn rate:
AgeGroup
Young      7.5
Mid       12.1
Senior    34.0
Elder     56.2
Old       24.8
Name: Exited, dtype: float64

IsHighRiskProduct churn rate:
IsHighRiskProduct
0    18.2
1    85.9
Name: Exited, dtype: float64

IsInactiveSenior churn rate:
IsInactiveSenior
0    17.2
1    82.2
Name: Exited, dtype: float64

IsFemaleGermany churn rate:
IsFemaleGermany
0    18.1
1    37.6
Name: Exited, dtype: float64
